# 📘 Khóa học: HR Analyst với SQL + PostgreSQL (7 buổi)

## 🎯 Tổng quan chương trình

- **Buổi 1:** 🚀 Giới thiệu dự án
- **Buổi 2:** 🗄️ Schema & Setup database
- **Buổi 3:** 👥 Employee Overview Analysis
- **Buổi 4:** 📊 Turnover Analysis
- **Buổi 5:** 💰 Salary Analysis
- **Buổi 6:** 📈 Engagement & Advanced Analysis
- **Buổi 7:** ✅ Quiz HR Analytics SQL

---

# 📊 Buổi 4: Turnover Analysis

## 🎯 Mục tiêu buổi này
✅ Sử dụng DATE functions (EXTRACT, AGE) để tính thời gian

✅ Dùng CASE WHEN để phân loại loại nghỉ việc (Voluntary vs Involuntary)

✅ Tính churn rate (tỷ lệ rời bỏ) theo thời gian

✅ Phân tích nguyên nhân rời bỏ

✅ Thực hành 5 bài tập đơn giản + 4 bài nâng cao

✅ Ôn tập 5 quiz từ cơ bản → nâng cao

---

## 🌏 Turnover Analysis là gì?

**Turnover Analysis** (Phân tích tỷ lệ rời bỏ) là phân tích số lượng nhân viên rời khỏi công ty. Đây là chỉ số **cực kỳ quan trọng** trong HR vì:

### 📊 Tại sao Turnover quan trọng?
- **Chi phí cao:** Thay thế 1 nhân viên = 50-200% lương thường niên (tuyển, đào tạo, mất năng suất)
- **Mất kiến thức:** Nhân viên có kinh nghiệm rời = mất IP, process, customer relationships
- **Ảnh hưởng tinh thần:** Turnover cao → morale thấp → more turnover (vòng lặp)
- **Uy tín công ty:** High turnover = khó tuyển người giỏi

### 📈 Turnover Rate bình thường:
- **Tech startups:** 15-25% (do thay đổi nhanh, cơ hội khác)
- **Corporate:** 10-15% (ổn định hơn)
- **Retail/F&B:** 25-50% (công việc seasonal, lương thấp)
- **Công ty lớn Việt Nam:** 20-30% (nhiều việc nhưng lương cạnh tranh)

### ❓ Các câu hỏi thường được hỏi
- Năm nay rời bỏ bao nhiêu nhân viên?
- Phòng nào có turnover cao nhất?
- Người nào rời (tình nguyện vs bị sa thải)?
- Lý do rời là gì? (tìm việc tốt hơn, lương thấp, quản lý tệ...)
- Nhân viên trẻ hay cũ rời nhiều hơn?

---

## 🔍 BƯỚC 1: DATE Functions - EXTRACT & AGE

**DATE Functions** dùng để làm việc với ngày tháng.
Trong phân tích Turnover ta cần:

- Khi nhân viên rời công ty → leaving_date

- Nhân viên làm bao lâu → hired_date → leaving_date

### 1.1 EXTRACT - Lấy năm/tháng từ ngày
```sql
SELECT 
  emp_id,
  leaving_date,
  EXTRACT(YEAR FROM leaving_date) AS leave_year,
  EXTRACT(MONTH FROM leaving_date) AS leave_month
FROM leavers
LIMIT 5;
```
**Kỳ vọng:** Tách năm/tháng rời bỏ từ ngày.

![img4-1.png](images/images_buoi4/img4-1.png)

### 1.2 AGE - Tính khoảng thời gian
```sql
SELECT 
  l.emp_id,
  e.hired_date,
  l.leaving_date,
  AGE(l.leaving_date, e.hired_date) AS tenure
FROM leavers l
JOIN employees e 
ON l.emp_id = e.emp_id
LIMIT 5;
```
**Kỳ vọng:** Tính số năm/tháng giữa 2 ngày (tenure = thời gian làm việc).

![img4-2.png](images/images_buoi4/img4-2.png)

### 1.3 Kết hợp AGE + EXTRACT
```sql
SELECT 
  l.emp_id,
  EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) AS years_tenure,
  EXTRACT(MONTH FROM AGE(l.leaving_date, e.hired_date)) AS months_tenure
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
LIMIT 5;
```
**Kỳ vọng:** Tách năm/tháng từ tenure (có thể được nói: "A rời sau 3 năm 5 tháng").

![img4-3.png](images/images_buoi4/img4-3.png)

**Giải thích:**
- `EXTRACT(YEAR FROM ...)` lấy năm từ ngày hoặc interval
- `AGE(date1, date2)` tính interval giữa 2 ngày (date1 - date2)
- Interval có thể extract tháng, ngày, v.v.

---

## 🔍 BƯỚC 2: CASE WHEN - Phân loại rời bỏ

**CASE WHEN** dùng để phân loại dữ liệu.

Trong HR Analytics thường phân loại:

- Voluntary → nhân viên tự nghỉ

- Involuntary → công ty cho nghỉ

Trong dataset này đã có sẵn cột exit_type.

### 2.1 Cơ bản: Tình nguyện vs Sa thải
```sql
SELECT 
  emp_id,
  leaving_date,
  reason,
  exit_type,
  CASE
    WHEN exit_type = 'Voluntary' THEN 'Employee Decision'
    WHEN exit_type = 'Involuntary' THEN 'Company Decision'
    WHEN exit_type = 'Retirement' THEN 'Retirement'
    ELSE 'Other'
  END AS separation_category
FROM leavers
LIMIT 10;
```
**Kỳ vọng:** Mỗi dòng ghi Voluntary/Involuntary/Unknown.

![img4-4.png](images/images_buoi4/img4-4.png)

### 2.2 Phân loại mức độ (High Risk vs Low Risk)
```sql
SELECT 
  l.emp_id,
  EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) AS years_tenure,
  CASE
    WHEN EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) < 1 THEN 'High Risk (<1 year)'
    WHEN EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) < 3 THEN 'Medium Risk (1-3 years)'
    ELSE 'Low Risk (>3 years)'
  END AS risk_category
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
LIMIT 10;
```
**Kỳ vọng:** Nhân viên rời sớm (< 1 năm) = High Risk.

![img4-5.png](images/images_buoi4/img4-5.png)

**Giải thích:**
- `IN ('val1', 'val2', ...)` kiểm tra nếu giá trị thuộc danh sách
- `WHEN ... THEN ...` là điều kiện và giá trị trả về
- `ELSE` là trường hợp mặc định nếu không match

---

## 🔍 BƯỚC 3: Churn Rate Calculation

**Churn Rate** = (Số nhân viên rời / Tổng nhân viên) × 100 %

### 3.1 Churn rate toàn công ty
```sql
SELECT 
  COUNT(DISTINCT l.emp_id) AS total_leavers,
  COUNT(DISTINCT e.emp_id) AS total_employees,
  ROUND(
    (COUNT(DISTINCT l.emp_id)::NUMERIC /
    COUNT(DISTINCT e.emp_id)) * 100,
    2
  ) AS churn_rate_pct
FROM leavers l
CROSS JOIN employees e;
```
**Kỳ vọng:** Ví dụ 100 rời / 1000 nhân viên = 10.00% churn rate.

![img4-6.png](images/images_buoi4/img4-6.png)

**Giải thích:**

- EXTRACT(YEAR FROM leaving_date) lấy năm nhân viên rời công ty.

- COUNT(DISTINCT l.emp_id) đếm số nhân viên rời trong năm đó.

- COUNT(DISTINCT e.emp_id) lấy tổng số nhân viên toàn công ty.

- (leavers / total_employees) * 100 tính churn rate (%).

- GROUP BY leave_year để tính churn rate theo từng năm.

- ORDER BY leave_year sắp xếp theo thời gian.

### 3.2 Churn rate theo năm
```sql
SELECT 
  EXTRACT(YEAR FROM l.leaving_date) AS leave_year,
  COUNT(DISTINCT l.emp_id) AS leavers,
  ROUND(
    (COUNT(DISTINCT l.emp_id)::NUMERIC /
    COUNT(DISTINCT e.emp_id)) * 100,
    2
  ) AS churn_rate_pct
FROM leavers l
CROSS JOIN employees e
GROUP BY EXTRACT(YEAR FROM l.leaving_date)
ORDER BY leave_year;
```
**Kỳ vọng:** 

![img4-7.png](images/images_buoi4/img4-7.png)

**Giải thích:** 

- JOIN departments để biết nhân viên thuộc phòng ban nào.

- LEFT JOIN leavers giúp giữ tất cả nhân viên, kể cả người chưa rời.

- COUNT(DISTINCT l.emp_id) đếm số nhân viên đã rời trong phòng ban.

- COUNT(DISTINCT e.emp_id) đếm tổng nhân viên của phòng ban.

- (leavers / total_emps) * 100 tính churn rate của phòng ban.

- ORDER BY churn_rate_pct DESC hiển thị phòng ban có churn cao nhất trước.


### 3.3 Churn rate theo phòng ban
```sql
SELECT 
  d.dept_name,
  COUNT(DISTINCT l.emp_id) AS leavers,
  COUNT(DISTINCT e.emp_id) AS total_emps,
  ROUND(
    COUNT(DISTINCT l.emp_id)::NUMERIC 
    / COUNT(DISTINCT e.emp_id) * 100,
    2
  ) AS churn_rate_pct
FROM employees e
LEFT JOIN leavers l 
  ON e.emp_id = l.emp_id
JOIN departments d 
  ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY churn_rate_pct DESC;
```
**Kỳ vọng:** 

![img4-8.png](images/images_buoi4/img4-8.png)

**Giải thích:**
- `::NUMERIC` ép kiểu số thập phân để chia chính xác
- `CROSS JOIN` để lấy tổng employees
- `GROUP BY EXTRACT(YEAR FROM ...)` để nhóm theo năm

---

## 🔍 BƯỚC 4: Phân tích chi tiết Voluntary vs Involuntary

### 4.1 Tỉ lệ nghỉ tình nguyện với nghỉ sa thải
```sql
SELECT 
  exit_type,
  COUNT(*) AS count,
  ROUND(
    COUNT(*)::NUMERIC / SUM(COUNT(*)) OVER () * 100,
    2
  ) AS pct_of_total
FROM leavers
GROUP BY exit_type
ORDER BY count DESC;
```
**Kỳ vọng:** 

![img4-9.png](images/images_buoi4/img4-9.png)

**Giải thích:** `SUM(COUNT(*)) OVER ()` -> đây là window function. Nó tính tổng số nhân viên rời công ty trên toàn bộ bảng. `OVER ()` nghĩa là tính trên toàn bộ dataset, không chia nhóm. -> `::NUMERIC` là type casting trong PostgreSQL. Chuyển số nguyên sang số thập phân để tránh phép chia bị làm tròn.
Ví dụ:
|exit_type|count|
|---|---|
|Voluntary|80|
|Involuntary|20|

- SUM(COUNT(*)) OVER () = 100

### 4.2 Phòng ban có nhiều người rời nhất
```sql
SELECT 
  d.dept_name,
  COUNT(l.emp_id) AS leavers
FROM leavers l
JOIN employees e 
ON l.emp_id = e.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY leavers DESC;
```
**Kỳ vọng:** 

![img4-10.png](images/images_buoi4/img4-10.png)

### 4.3 Lý do nghỉ việc phổ biến nhất ?
```sql
SELECT 
  reason,
  COUNT(*) AS count,
  ROUND(
    AVG(
      EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date))
    ),
    1
  ) AS avg_tenure_years
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
GROUP BY reason
ORDER BY count DESC;
```
**Kỳ vọng:** 

![img4-11.png](images/images_buoi4/img4-11.png)


---

## 💪 BƯỚC 5: 5 Bài tập đơn giản

### Bài 1: Hiển thị 10 nhân viên rời công ty gần đây nhất.
**Yêu cầu:** Viết query hiển thị 10 nhân viên rời công ty gần đây nhất.

**Đáp án:**
```sql
SELECT 
  emp_id,
  leaving_date,
  reason,
  exit_type
FROM leavers
ORDER BY leaving_date DESC
LIMIT 10;
```

![img4-12.png](images/images_buoi4/img4-12.png)

**Giải thích:** 

- ORDER BY leaving_date DESC → rời gần nhất lên đầu

- LIMIT 10 → lấy 10 dòng đầu

### Bài 2: Số nhân viên rời theo phòng ban
**Yêu cầu:** Viết query đếm số nhân viên rời theo tháng phòng ban.

**Đáp án:**
```sql
SELECT 
  d.dept_name,
  COUNT(l.emp_id) AS total_leavers
FROM leavers l
JOIN employees e 
ON l.emp_id = e.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY total_leavers DESC;
```

![img4-13.png](images/images_buoi4/img4-13.png)

**Giải thích:** 

- JOIN để biết nhân viên thuộc phòng ban nào

- GROUP BY dept_name → gom theo phòng

- COUNT() → đếm số người rời

---

### Bài 3: Tenure của người rời
**Yêu cầu:** Viết query hiển thị nhân viên rời và số năm làm việc..

**Đáp án:**
```sql
SELECT 
  l.emp_id,
  e.hired_date,
  l.leaving_date,
  EXTRACT(YEAR FROM AGE(l.leaving_date, e.hired_date)) AS tenure_years
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
ORDER BY tenure_years DESC;
```

![img4-14.png](images/images_buoi4/img4-14.png)

**Giải thích:** `CASE WHEN ... IN ... THEN` phân loại lý do rời. `AGE()` tính khoảng thời gian giữa 2 ngày. `EXTRACT(YEAR FROM ...)` lấy số năm làm việc.

---

### Bài 4: Lương trung bình của nhân viên đã rời
**Yêu cầu:** Viết query tính lương trung bình của nhân viên đã rời công ty.

**Đáp án:**
```sql
SELECT 
  ROUND(AVG(s.base_salary),0) AS avg_salary
FROM leavers l
JOIN salaries s
ON l.emp_id = s.emp_id;
```

![img4-15.png](images/images_buoi4/img4-15.png)

**Giải thích:** `JOIN salaries` để lấy lương. `AVG(base_salary)` tính trung bình. `ROUND()` làm tròn kết quả.

---

### Bài 5: Phòng ban có nhiều Involuntary Exit nhất
**Yêu cầu:** Tìm phòng ban có nhiều nhân viên bị cho nghỉ việc nhất.

**Đáp án:**
```sql
SELECT 
  d.dept_name,
  COUNT(*) AS involuntary_leavers
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
WHERE l.exit_type = 'Involuntary'
GROUP BY d.dept_name
ORDER BY involuntary_leavers DESC;
```
![img4-16.png](images/images_buoi4/img4-16.png)

**Giải thích:** `WHERE exit_type = 'Involuntary'` → lọc loại nghỉ việc. `COUNT(*)` → đếm số trường hợp

---

## 🚀 BƯỚC 6: 4 Bài tập nâng cao

### Bài 6: Top 3 phòng ban churn cao nhất
**Yêu cầu:** Viết query tìm ra Top 3 phòng ban churn cao nhất.

**Đáp án:**
```sql
SELECT 
  d.dept_name,
  COUNT(l.emp_id) AS leavers,
  COUNT(e.emp_id) AS total_employees,
  ROUND(
    COUNT(l.emp_id)::NUMERIC /
    COUNT(e.emp_id) * 100,
    2
  ) AS churn_rate_pct
FROM employees e
LEFT JOIN leavers l
ON e.emp_id = l.emp_id
JOIN departments d
ON e.dept_id = d.dept_id
GROUP BY d.dept_name
ORDER BY churn_rate_pct DESC
LIMIT 3;
```
![img4-17.png](images/images_buoi4/img4-17.png)

**Giải thích:** `LEFT JOIN` giữ toàn bộ nhân viên. `COUNT(l.emp_id)` → số người rời. `(leavers / total_employees) * 100` → churn rate

---

### Bài 7: Xu hướng nghỉ việc theo quý
**Yêu cầu:** Viết query tìm ra Xu hướng nghỉ việc theo quý.

**Đáp án:**
```sql
SELECT
  DATE_TRUNC('quarter', leaving_date) AS leave_quarter,
  COUNT(*) AS total_leavers
FROM leavers
GROUP BY leave_quarter
ORDER BY leave_quarter;
```

![img4-18.png](images/images_buoi4/img4-18.png)

**Giải thích:** `DATE_TRUNC('quarter')` gom dữ liệu theo quý.

---

### Bài 8: Risk group - Rời sớm quá
**Yêu cầu:** Viết query tìm nhân viên rời trong vòng < 1 năm (High Risk = may bỏ ngang).

**Đáp án:**
```sql
SELECT 
  COUNT(
    CASE 
      WHEN AGE(l.leaving_date, e.hired_date) < INTERVAL '1 year'
      THEN 1
    END
  ) AS early_leavers,
  COUNT(*) AS total_leavers,
  ROUND(
    COUNT(
      CASE 
        WHEN AGE(l.leaving_date, e.hired_date) < INTERVAL '1 year'
        THEN 1
      END
    )::NUMERIC / COUNT(*) * 100,
    2
  ) AS early_attrition_pct
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id;
```
![img4-19.png](images/images_buoi4/img4-19.png)

**Giải thích:** `AGE()` tính thời gian làm việc. `< INTERVAL '1 year'` → xác định rời sớm chia cho total_leavers để ra tỷ lệ %

---

### Bài 9: Turnover theo nhóm tenure
**Yêu cầu:** Phân loại nhân viên rời theo thời gian làm việc. Sau đó đếm số người rời trong từng nhóm:

- < 1 năm

- 1–3 năm

- 3–5 năm

- > 5 năm

**Đáp án:**
```sql
SELECT
  CASE
    WHEN AGE(l.leaving_date, e.hired_date) < INTERVAL '1 year'
      THEN '<1 year'
    WHEN AGE(l.leaving_date, e.hired_date) < INTERVAL '3 years'
      THEN '1-3 years'
    WHEN AGE(l.leaving_date, e.hired_date) < INTERVAL '5 years'
      THEN '3-5 years'
    ELSE '>5 years'
  END AS tenure_group,
  COUNT(*) AS total_leavers
FROM leavers l
JOIN employees e
ON l.emp_id = e.emp_id
GROUP BY tenure_group
ORDER BY total_leavers DESC;
```
![img4-20.png](images/images_buoi4/img4-20.png)

**Giải thích:** `AGE()` → tính thời gian làm việc. `CASE WHEN` → chia tenure thành các nhóm. `GROUP BY` → đếm số người rời trong mỗi nhóm

---

## 📝 BƯỚC 7: 5 Câu hỏi Quiz

### Quiz 1 (Cơ bản): EXTRACT Year
**Câu hỏi:** Để lấy năm từ leave_date, query nào đúng?
- A) `SELECT YEAR(leave_date) FROM leavers`
- B) `SELECT EXTRACT(YEAR FROM leave_date) FROM leavers` ✅
- C) `SELECT DATE_PART('year', leave_date) FROM leavers`
- D) `SELECT TO_CHAR(leave_date, 'YYYY') FROM leavers`

**Đáp án:** B ✅

**Giải thích:** PostgreSQL dùng `EXTRACT(YEAR FROM ...)`. Câu C (`DATE_PART`) cũng đúng nhưng B là tiêu chuẩn SQL. Câu A (YEAR) là MySQL, Câu D (TO_CHAR) trả về text.

---

### Quiz 2 (Cơ bản): AGE Function
**Câu hỏi:** `AGE(leave_date, hired_date)` tính cái gì?
- A) Số ngày giữa 2 ngày
- B) Khoảng thời gian (interval) giữa 2 ngày ✅
- C) Hiệu số tuyệt đối
- D) Lỗi syntax

**Đáp án:** B ✅

**Giải thích:** `AGE(date1, date2)` trả về interval (có năm, tháng, ngày). Muốn lấy số ngày dùng `date1 - date2`.

---

### Quiz 3 (Trung cấp): CASE WHEN
**Câu hỏi:** Để phân loại Voluntary vs Involuntary dựa trên `reason` column, query nào đúng?
- A) `SELECT CASE reason IN ('Resigned') THEN 'Voluntary' END FROM leavers`
- B) `SELECT CASE WHEN reason IN ('Resigned', 'Better Opportunity') THEN 'Voluntary' ELSE 'Involuntary' END FROM leavers` ✅
- C) `SELECT IF(reason = 'Resigned', 'Voluntary', 'Involuntary') FROM leavers`
- D) `SELECT reason AS 'Voluntary' FROM leavers WHERE reason IN ('Resigned')`

**Đáp án:** B ✅

**Giải thích:** Câu B dùng `CASE WHEN ... THEN ... ELSE ... END` đúng syntax PostgreSQL. Câu A thiếu WHEN. Câu C là MySQL IF. Câu D sai logic (rename column).

---

### Quiz 4 (Trung cấp): Churn Rate Formula
**Câu hỏi:** Churn rate được tính như thế nào?
- A) Số leavers / số employees
- B) (Số leavers / số employees) × 100 ✅
- C) Số leavers × 100
- D) Số leavers - số employees

**Đáp án:** B ✅

**Giải thích:** Churn rate là tỷ lệ (%) nên phải × 100. Ví dụ: 50 rời / 500 NV = 0.1 × 100 = 10%.

---

### Quiz 5 (Nâng cao): Phân loại Tenure
**Câu hỏi:** Để phân loại nhân viên theo nhóm tenure (<1 year, 1–3 years, >3 years) nên dùng SQL nào?
- A) CASE WHEN ... THEN ... ✅
- B) GROUP BY 
- C) LIMIT
- D) DISTINCT

**Đáp án:** B ✅

**Giải thích:** `CASE WHEN` cho phép tạo nhóm dữ liệu mới dựa trên điều kiện.

---

## 📖 Tài liệu tham khảo & Links

### PostgreSQL DATE Functions & CASE WHEN
- 🔗 [DATE Functions - PostgreSQL Docs](https://www.postgresql.org/docs/current/functions-datetime.html)
- 🔗 [EXTRACT & AGE Functions - PostgreSQL Manual](https://www.postgresql.org/docs/current/functions-datetime.html#FUNCTIONS-DATETIME-EXTRACT)
- 🔗 [CASE Expression - PostgreSQL Docs](https://www.postgresql.org/docs/current/sql-expressions.html#SQL-CASE)

### Employee Turnover Analytics
- 🔗 [Turnover Rate Calculation - BLS](https://www.bls.gov/opub/hom/pdf/homch5.pdf)
- 🔗 [Voluntary vs Involuntary Separation - SHRM](https://www.shrm.org/)
- 🔗 [Churn Rate Definition & Metrics - HR.com](https://www.hr.com/)

### Cost of Turnover & Retention
- 🔗 [Cost of Employee Turnover - WorkForce Institute](https://www.workforceinstitute.org/)
- 🔗 [Retention Strategies - McKinsey HR](https://www.mckinsey.com/industries/human-capital/)

---

## 🎓 Kết luận Buổi 4

✅ **Bạn vừa hoàn thành:**
- Dùng DATE functions (EXTRACT, AGE) để tính thời gian rời bỏ
- Sử dụng CASE WHEN để phân loại Voluntary vs Involuntary
- Tính churn rate (%) theo thời gian & phòng ban
- Phân tích chi tiết lý do rời bỏ
- Thực hành 5 bài tập đơn giản + 4 nâng cao
- Ôn tập 5 quiz từ cơ bản → nâng cao

📅 **Buổi 5**: Salary Analysis - Phân tích lương
- Window Functions (RANK, ROW_NUMBER, LAG)
- Gender pay gap (công bằng lương)
- Salary history & trends
- Percentile analysis

---

💡 **Ghi chú quan trọng:**
- DATE functions (EXTRACT, AGE) là bắt buộc để phân tích thời gian rời bỏ
- CASE WHEN giúp phân loại logic phức tạp (Voluntary vs Involuntary)
- Churn rate (%) là KPI chính để track health của company
- Turnover cao trong năm `< 1 năm = mất investment tuyển/đào tạo
- Luôn tính churn rate theo phòng để identify problem areas

🚀 **Hãy thực hành các bài tập trên để quen tay với SQL và HR Analytics!**